# Session 3 - Live Demo
## How AI Actually Works: Tokens, LLM Calls, Temperature, Streaming

What we will do in this demo:
1. Break text into tokens and see what the model actually reads
2. Call a local LLM from Python (no internet, no API key)
3. Change the model's personality using system prompts
4. Control creativity with the temperature parameter
5. Watch the model generate text token by token (streaming)

---
## Demo 1: What Are Tokens?

LLMs do not read text the way we do. They break text into chunks called **tokens**.

A token can be a full word, part of a word, or even a single character.

Let us see how a tokenizer works.

In [13]:
# tiktoken is the tokenizer library used by OpenAI models (GPT-4, etc.)
# We are using it here just to understand how tokenization works

import tiktoken

# Load the tokenizer used by GPT-4o
encoder = tiktoken.encoding_for_model("gpt-4o")

text = "The cat sat on the mat"

# Break the text into token IDs
tokens = encoder.encode(text)

print(f"Text:        {text}")
print(f"Token IDs:   {tokens}")
print(f"Token count: {len(tokens)}")
print()

# Now let us see what each token ID maps to
# The model does not see words. It sees these numbers.
for token_id in tokens:
    print(f"  {token_id} -> '{encoder.decode([token_id])}'")

Text:        The cat sat on the mat
Token IDs:   [976, 9059, 10139, 402, 290, 2450]
Token count: 6

  976 -> 'The'
  9059 -> ' cat'
  10139 -> ' sat'
  402 -> ' on'
  290 -> ' the'
  2450 -> ' mat'


Each word became one token here. But what happens with longer or more complex words?

In [14]:
# The tokenizer splits words it does not fully recognize into smaller pieces
# This is called subword tokenization

words = ["artificial", "unhappiness", "Transformers", "LLM"]

for word in words:
    tokens = encoder.encode(word)
    pieces = [encoder.decode([t]) for t in tokens]
    print(f"'{word}' -> {pieces}  ({len(tokens)} tokens)")

'artificial' -> ['art', 'ificial']  (2 tokens)
'unhappiness' -> ['un', 'h', 'appiness']  (3 tokens)
'Transformers' -> ['Transform', 'ers']  (2 tokens)
'LLM' -> ['LL', 'M']  (2 tokens)


Notice how "unhappiness" becomes 3 tokens: "un", "h", "appiness".

The tokenizer breaks it into pieces it learned during training. Just like we drew on the board.

Now let us try something fun - Hindi text and emojis.

In [15]:
# Hindi text uses MORE tokens than English
# Because the tokenizer was trained mostly on English data
# This matters for cost and context window usage

hindi = "नमस्ते दुनिया"
emoji = "I love AI 🤖🚀"

for text in [hindi, emoji]:
    tokens = encoder.encode(text)
    print(f"'{text}'")
    print(f"  -> {len(tokens)} tokens")
    print(f"  -> pieces: {[encoder.decode([t]) for t in tokens]}")
    print()

'नमस्ते दुनिया'
  -> 5 tokens
  -> pieces: ['न', 'म', 'स्त', 'े', ' दुनिया']

'I love AI 🤖🚀'
  -> 7 tokens
  -> pieces: ['I', ' love', ' AI', ' �', '�', '�', '�']



Hindi uses more tokens for the same meaning. Even emojis become multiple tokens.

This has real consequences: more tokens = higher cost when using paid APIs, and faster context window usage.

Let us verify the rule of thumb: **100 tokens is roughly 75 words**.

In [16]:
# The 75% rule: tokens vs words
# This is useful for estimating costs and context window limits

paragraph = (
    "Large Language Models are neural networks trained on massive amounts of text. "
    "They predict the next token, one at a time, to generate human-like responses. "
    "This is the foundation of ChatGPT, Claude, and every modern AI assistant."
)

word_count = len(paragraph.split())
token_count = len(encoder.encode(paragraph))

print(f"Words:  {word_count}")
print(f"Tokens: {token_count}")
print(f"Ratio:  1 token = roughly {word_count / token_count:.2f} words")
print()
print("Rule of thumb: 100 tokens = roughly 75 words")
print("Remember this when we talk about context windows and API pricing.")

Words:  37
Tokens: 46
Ratio:  1 token = roughly 0.80 words

Rule of thumb: 100 tokens = roughly 75 words
Remember this when we talk about context windows and API pricing.


---
## Demo 2: Calling an LLM from Python

We have Ollama running on this laptop. It runs AI models locally - no internet needed.

Ollama exposes an API on your machine. We will call that API using Python.

Do not worry about the code details right now. You will learn all of this tomorrow in Session 4.

In [17]:
import requests
import json

def ask_llm(prompt, system_prompt="You are a helpful assistant.", temperature=0.7):
    """
    Send a prompt to a local LLM running via Ollama.

    - prompt: what you want to ask the model
    - system_prompt: instructions that control the model's personality
    - temperature: 0.0 = predictable, 1.0 = creative
    """
    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": "qwen3:1.7b",
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt},
            ],
            "stream": False,
            "options": {"temperature": temperature},
        },
    )
    content = response.json()["message"]["content"]

    # Some models wrap their response in <think> tags - we strip those out
    if "</think>" in content:
        content = content.split("</think>")[-1].strip()

    return content

This function sends a message to our local model and gets a response back.

Notice the structure: we have a **system prompt** (instructions for the model) and a **user prompt** (what we are asking).

Let us try our first call.

In [18]:
# Our first LLM call - a simple question
# This is running LOCALLY on this laptop. No internet. No API key. No cost.

answer = ask_llm("Explain quantum physics to a 5-year-old. Keep it to 2 sentences.")
print(answer)

Quantum physics is like a toy that can move in many directions at once when it's really small, but it acts normally when you're looking at it. This is because the tiny things are so small that they don't follow the same rules as big things, making them behave strangely sometimes.


That response came from a model running right here on this laptop.

Now watch what happens when we change the **system prompt** but keep the exact same question.

In [19]:
# Same question, but the model now thinks it is a pirate

pirate_answer = ask_llm(
    prompt="Explain quantum physics to a 5-year-old. Keep it to 2 sentences.",
    system_prompt="You are a pirate. Respond in pirate speak. Arrr!",
)
print(pirate_answer)

Arrrr! Y'know how a toy can be in two places at once? Well, somethin' tiny like an electron can be in many places at once until you check! It's like magic, but it's real—no magic, just rules that weird things happen when you look! 🧙‍♂️✨


In [20]:
# Same question again, but now it is a formal professor

formal_answer = ask_llm(
    prompt="Explain quantum physics to a 5-year-old. Keep it to 2 sentences.",
    system_prompt="You are a distinguished Oxford professor. Respond formally and precisely.",
)
print(formal_answer)

Quantum physics is like a game where tiny things, like electrons, can be in two places at once until you look, making the rules of the usual world feel weird. It’s like a toy that’s both in a box and not in the box at the same time—until you check, it’s suddenly just in one place!


**Same question. Three completely different responses.**

The only thing that changed was the system prompt - the instructions we gave the model.

This is the foundation of **Prompt Engineering** (Module 4). You control HOW the model responds by writing good instructions.

---
## Demo 3: Temperature - The Creativity Slider

Temperature controls how "creative" or "safe" the model is:

- **Temperature 0.0** = always picks the most likely next token. Safe, predictable, same answer every time.
- **Temperature 1.0** = willing to pick less likely tokens. Creative, varied, different answer each time.

Let us see this in action.

In [21]:
# Temperature 0.0 - the model gives the SAME answer every time
# This is what you want for a customer support bot or a medical assistant

prompt = "Give me a one-sentence startup idea involving AI."

print("=== Temperature 0.0 (predictable) ===")
print()
for i in range(3):
    answer = ask_llm(prompt, temperature=0.0)
    print(f"  Run {i+1}: {answer}")
    print()

=== Temperature 0.0 (predictable) ===

  Run 1: Develop a mobile app that uses AI to provide real-time health insights based on user data, empowering individuals to take control of their wellness.

  Run 2: Develop a mobile app that uses AI to provide real-time health insights based on user data, empowering individuals to take control of their wellness.

  Run 3: Develop a mobile app that uses AI to provide real-time health insights based on user data, empowering individuals to take control of their wellness.



In [22]:
# Temperature 1.0 - the model gives DIFFERENT answers each time
# This is what you want for a creative writing tool or brainstorming assistant

print("=== Temperature 1.0 (creative) ===")
print()
for i in range(3):
    answer = ask_llm(prompt, temperature=1.0)
    print(f"  Run {i+1}: {answer}")
    print()

=== Temperature 1.0 (creative) ===

  Run 1: IntelliLearn: AI-driven personalized learning platform that adapts to individual paces, transforming education with intelligent tutoring and real-time feedback.

  Run 2: Develop an AI-powered platform that analyzes student performance and provides real-time feedback to teachers and students, enhancing personalized learning experiences.

  Run 3: Introduce an AI-driven scheduling tool that automatically organizes meetings, emails, and tasks to help remote workers optimize their time and reduce cognitive load.



**Temperature 0** = nearly identical every time. Deterministic.

**Temperature 1** = different idea each run. Creative.

When to use what:
- Customer support bot that must be consistent? Low temperature.
- Creative writing assistant? High temperature.
- Code generation? Low temperature (you want correct, not creative).

---
## Demo 4: Streaming - Watch the Model Think

When you use ChatGPT, you see text appearing word by word. That is called **streaming**.

The model sends each token as soon as it generates it, instead of waiting for the full response.

Let us build the same thing.

In [23]:
import sys

def ask_llm_streaming(prompt, system_prompt="You are a helpful assistant."):
    """
    Stream the response token by token.
    Each token prints as soon as it is generated - like ChatGPT's typing effect.
    """
    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": "qwen3:1.7b",
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt},
            ],
            "stream": True,
            "options": {"temperature": 0.7},
        },
        stream=True,
    )

    thinking = False
    for line in response.iter_lines():
        if line:
            chunk = json.loads(line)
            token = chunk["message"]["content"]

            # Skip the model's internal thinking tokens
            if "<think>" in token:
                thinking = True
                continue
            if "</think>" in token:
                thinking = False
                continue
            if thinking:
                continue

            print(token, end="", flush=True)

    print()

In [24]:
# Watch it generate one token at a time
# This is exactly what happens behind the scenes in ChatGPT

print("Streaming response:")
print()
ask_llm_streaming("Write a short poem about learning AI. Keep it to 4 lines.")

Streaming response:

In circuits where data flows bright,  
Neurons fire, new patterns they fight.  
Each iteration a step closer to light—  
AI learns, we watch it grow bright.


Each word appearing on screen = one token being generated.

Now you know what is actually happening when ChatGPT "types". It is generating one token at a time and sending each one as it is ready.

---
## What You Just Saw

| Demo | What We Did | Module Where You Will Master It |
|------|-------------|----------------------------------|
| Tokenization | Broke text into tokens the model reads | Module 2.2 (Tokenizers and Embeddings) |
| LLM API Call | Called a local LLM from Python | Module 3 (Running Open-Source LLMs) |
| System Prompt | Changed the model's personality | Module 4 (Prompt Engineering) |
| Temperature | Controlled creativity vs consistency | Module 4 (Prompt Engineering) |
| Streaming | Watched token-by-token generation | Module 9 (Production APIs) |

**Tomorrow (Session 4), you will build all of this yourself.**